In [1]:
import torch
import torch.nn as nn
import numpy as np



In [8]:
# 约定: (B, T, E, H) 分别表示 批次/序列长度/输入维度/隐藏维度
B, E, H = 1, 128, 3

def prepare_inputs():
    """
    使用 NumPy 准备输入数据
    使用示例句子: "播放 周杰伦 的 《稻香》"
    构造最小词表和随机(可复现)词向量, 生成形状为 (B, T, E) 的输入张量。
    """
    np.random.seed(42)
    vocab = {"播放": 0, "周杰伦": 1, "的": 2, "《稻香》": 3}
    tokens = ["播放", "周杰伦", "的", "《稻香》"]
    ids = [vocab[t] for t in tokens]

    # 词向量表: (V, E)
    V = len(vocab)
    # 4 * 128 词向量表（包含所有4个词的向量）
    emb_table = np.random.randn(V, E).astype(np.float32)

    # 取出序列词向量并加上 batch 维度: (B, T, E)
    # 按特定顺序取出的序列（"播放", "周杰伦", "的", "《稻香》"）
    # vectors = emb_table[ids]
    # vectors[Nont] 给输入序列添加 batch 维度。
    x_np = emb_table[ids][None]
    return tokens, x_np

In [9]:
def manual_rnn_numpy(x_np, U_np, W_np):
    """
    使用 NumPy 手动实现 RNN(无偏置): h_t = tanh(U x_t + W h_{t-1})

    Args:
        x_np: (B, T, E)
        U_np: (E, H)
        W_np: (H, H)
    Returns:
        outputs: (B, T, H)
        final_h: (B, H)
    """
    # B 批次大小 1
    # T 序列长度 几个词 4
    B_local, T_local, _ = x_np.shape
    # 初始化 h_0 为零向量 1 * 3
    h_prev = np.zeros((B_local, H), dtype=np.float32)

    steps = []
    # 按时间步循环
    for t in range(T_local):
        # 取所有批次、第 t 个位置、所有特征
        x_t = x_np[:, t, :]
        # 核心公式实现
        # x_t @ U_np：输入变换 (B, E) × (E, H) → (B, H)
        # h_prev @ W_np：循环变换 (B, H) × (H, H) → (B, H)
        h_t = np.tanh(x_t @ U_np + h_prev @ W_np)
        steps.append(h_t)
        h_prev = h_t # 更新状态
    # 将 (T, B, H) 堆叠成 (B, T, H)
    return np.stack(steps, axis=1), h_prev

In [10]:
def pytorch_rnn_forward(x, U, W):
    rnn = nn.RNN(
        input_size=E,
        hidden_size=H,
        num_layers=1,
        nonlinearity='tanh',
        bias=False,
        batch_first=True,
        bidirectional=False,
    )
    with torch.no_grad():
        # PyTorch 内部存放的是转置后的权重
        rnn.weight_ih_l0.copy_(U.T)
        rnn.weight_hh_l0.copy_(W.T)
    y, h_n = rnn(x)
    return y, h_n.squeeze(0)


In [14]:
def main():
    _, x_np = prepare_inputs()

    # PyTorch 张量，用于 nn.RNN 模块
    x = torch.from_numpy(x_np).float()

    # 使用可学习参数 U, W（无偏置）
    torch.manual_seed(7)
    U = torch.randn(E, H)
    W = torch.randn(H, H)

    # --- 手写 RNN (使用 NumPy) ---
    U_np = U.detach().numpy()
    W_np = W.detach().numpy()

    print("--- 手写 RNN (NumPy) ---")
    out_manual_np, hT_manual_np = manual_rnn_numpy(x_np, U_np, W_np)
    print("输入形状:", x_np.shape)
    print("输入:", x_np)
    print("手写输出形状:", out_manual_np.shape)
    print("手写输出:", out_manual_np)
    print("手写最终隐藏形状:", hT_manual_np.shape)
    print("手写最终隐藏:", hT_manual_np)

    print("\n--- PyTorch nn.RNN ---")
    out_torch, hT_torch = pytorch_rnn_forward(x, U, W)
    print("模块输出形状:", out_torch.shape)
    print("模块最终隐藏形状:", hT_torch.shape)

    print("\n--- 对齐验证 ---")
    # 将 NumPy 结果转回 PyTorch 张量以进行比较
    out_manual = torch.from_numpy(out_manual_np)
    hT_manual = torch.from_numpy(hT_manual_np)

    print("逐步输出一致:", torch.allclose(out_manual, out_torch, atol=1e-6))
    print("最终隐藏一致:", torch.allclose(hT_manual, hT_torch, atol=1e-6))
    print("最后一步输出等于最终隐藏:", torch.allclose(out_torch[:, -1, :], hT_torch, atol=1e-6))


if __name__ == "__main__":
    main()

--- 手写 RNN (NumPy) ---
输入形状: (1, 4, 128)
输入: [[[ 0.49671414 -0.1382643   0.64768857  1.5230298  -0.23415338
   -0.23413695  1.5792128   0.7674347  -0.46947438  0.54256004
   -0.46341768 -0.46572974  0.24196227 -1.9132802  -1.7249179
   -0.5622875  -1.0128311   0.31424734 -0.9080241  -1.4123037
    1.4656488  -0.2257763   0.0675282  -1.4247482  -0.54438275
    0.11092259 -1.1509936   0.37569803 -0.6006387  -0.29169375
   -0.6017066   1.8522782  -0.01349723 -1.0577109   0.82254493
   -1.2208437   0.2088636  -1.9596701  -1.328186    0.19686124
    0.73846656  0.17136829 -0.11564828 -0.30110368 -1.478522
   -0.7198442  -0.46063876  1.0571222   0.3436183  -1.7630402
    0.32408398 -0.38508227 -0.676922    0.6116763   1.0309995
    0.93128014 -0.83921754 -0.3092124   0.33126342  0.9755451
   -0.47917423 -0.18565898 -1.1063349  -1.1962066   0.8125258
    1.35624    -0.07201012  1.0035329   0.361636   -0.6451197
    0.3613956   1.5380366  -0.03582604  1.5646436  -2.619745
    0.8219025   0.087